In [ ]:
!pip install datasets transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 452 kB 33.9 MB/s 
     |████████████████████████████████| 5.8 MB 57.6 MB/s 
     |████████████████████████████████| 182 kB 70.1 MB/s 
     |████████████████████████████████| 213 kB 73.7 MB/s 
     |████████████████████████████████| 132 kB 74.2 MB/s 
     |████████████████████████████████| 127 kB 64.7 MB/s 
     |████████████████████████████████| 7.6 MB 70.1 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.huggingface/token
Login successful


In [ ]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree       
Reading state information... Done
git-lfs is already the newest version (2.3.4-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-460
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 20 not upgraded.


In [ ]:
import transformers 
print(transformers.__version__)

4.25.1


In [ ]:
squad_v2 = True
model_checkpoint = 'bhadresh-savani/electra-base-squad2'
batch_size = 16

In [ ]:
from transformers import AutoTokenizer
    
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Downloading:   0%|          | 0.00/200 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/768 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset, load_metric, Dataset
from datasets.dataset_dict import DatasetDict

In [ ]:
from zipfile import ZipFile

with ZipFile('/content/train_data.zip', 'r') as zipobj:
  zipobj.extractall()
  print('Done')

Done


In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import ast

df = pd.read_csv('/content/train_data.csv')
df.drop(columns=['Answer_possible'], inplace=True)

In [ ]:
df['Paragraph'][0]

'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".'

In [ ]:
rows_to_rem = []
for idx in tqdm(df.index):
  val = len(tokenizer(df.loc[idx, 'Paragraph'], df.loc[idx, 'Question']).input_ids)
  if val > 512:
    rows_to_rem.append(idx)

print('\n')
print(str(len(rows_to_rem)) + 'rows removed')
df.drop(labels=rows_to_rem, inplace=True)
df.reset_index(drop=True, inplace=True)

100%|██████████| 75055/75055 [00:51<00:00, 1458.63it/s]



102rows removed


In [ ]:
val_themes = df['Theme'].value_counts(ascending=True)[:100].index
train_df = df[~df["Theme"].isin(val_themes)]
test_df = df[df["Theme"].isin(val_themes)]

In [ ]:
train_ans = []
test_ans = []

for idx in train_df.index:
  d = {}
  d['answer_start'] = ast.literal_eval(train_df.loc[idx]['Answer_start'])
  d['text'] = ast.literal_eval(train_df.loc[idx]['Answer_text'])
  train_ans.append(d)
for idx in test_df.index:
  d = {}
  d["answer_start"] = ast.literal_eval(test_df.loc[idx]["Answer_start"])
  d["text"] = ast.literal_eval(test_df.loc[idx]["Answer_text"])
  test_ans.append(d)

In [ ]:
train_data, test_data = {}, {}
test_data['id'] = test_df['Unnamed: 0'].values.tolist()
train_data['title'] = train_df['Theme'].values.tolist()
train_data['context'] = train_df['Paragraph'].values.tolist()
train_data['question'] = train_df['Question'].values.tolist()
train_data['answers'] = train_ans

test_data['id'] = test_df['Unnamed: 0'].values.tolist()
test_data['title'] = test_df['Theme'].values.tolist()
test_data['context'] = test_df['Paragraph'].values.tolist()
test_data['question'] = test_df['Question'].values.tolist()
test_data['answers'] = test_ans

In [ ]:
train_data = Dataset.from_dict(train_data)
test_data = Dataset.from_dict(test_data)

In [ ]:
train_data.to_json('train_data.json')
test_data.to_json('test_data.json')

Creating json from Arrow format:   0%|          | 0/62 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/14 [00:00<?, ?ba/s]

11686871

In [ ]:
datasets = DatasetDict({'train':train_data, 'validation':test_data})
datasets

DatasetDict({
    train: Dataset({
        features: ['title', 'context', 'question', 'answers'],
        num_rows: 61777
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 13176
    })
})

In [ ]:
datasets['train'][0]

{'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': "When did Beyonce leave Destiny's Child and become a solo singer?",
 'answers': {'answer_start': [526], 'text': ['2003']}}

In [ ]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
  assert num_examples <= len(dataset), "Can't pick more elements than there are in dataset."
  picks = []
  for _ in range(num_examples):
    pick = random.randint(0, len(dataset)-1)
    while pick in picks:
      pick = random.randint(0, len(dataset)-1)
    picks.append(pick)

  df = pd.DataFrame(dataset[picks])
  for column, typ in dataset.features.items():
    if isinstance(typ, ClassLabel):
      df[column] = df[column].transform(lambda i:typ.names[i])
    elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
      df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
  display(HTML(df.to_html()))

In [ ]:
show_random_elements(datasets["train"])

,title,context,question,answers
0,United_States_Air_Force,"Since 2005, the USAF has placed a strong focus on the improvement of Basic Military Training (BMT) for enlisted personnel. While the intense training has become longer, it also has shifted to include a deployment phase. This deployment phase, now called the BEAST, places the trainees in a surreal environment that they may experience once they deploy. While the trainees do tackle the massive obstacle courses along with the BEAST, the other portions include defending and protecting their base of operations, forming a structure of leadership, directing search and recovery, and basic self aid buddy care. During this event, the Military Training Instructors (MTI) act as mentors and enemy forces in a deployment exercise.",What is the deployment phase of BMT called?,"{'answer_start': [258], 'text': ['BEAST']}"
1,Kanye_West,"On January 5, 2012, West announced his establishment of the creative content company DONDA, named after his late mother Donda West. In his announcement, West proclaimed that the company would ""pick up where Steve Jobs left off""; DONDA would operate as ""a design company which will galvanize amazing thinkers in a creative space to bounce their dreams and ideas"" with the ""goal to make products and experiences that people want and can afford."" West is notoriously secretive about the company's operations, maintaining neither an official website nor a social media presence. In stating DONDA's creative philosophy, West articulated the need to ""put creatives in a room together with like minds"" in order to ""simplify and aesthetically improve everything we see, taste, touch, and feel."". Contemporary critics have noted the consistent minimalistic aesthetic exhibited throughout DONDA creative projects.",On what date did Kanye go public with his DONDA company?,"{'answer_start': [3], 'text': ['January 5, 2012']}"
2,Railway_electrification_system,"3 kV DC is used in Belgium, Italy, Spain, Poland, the northern Czech Republic, Slovakia, Slovenia, South Africa, Chile, and former Soviet Union countries (also using 25 kV 50 Hz AC). It was formerly used by the Milwaukee Road from Harlowton, Montana to Seattle-Tacoma, across the Continental Divide and including extensive branch and loop lines in Montana, and by the Delaware, Lackawanna & Western Railroad (now New Jersey Transit, converted to 25 kV AC) in the United States, and the Kolkata suburban railway (Bardhaman Main Line) in India, before it was converted to 25 kV 50 Hz AC.",What does the railway system of US use DC or AC?,"{'answer_start': [582], 'text': ['AC']}"
3,Gothic_architecture,"The term ""Gothic architecture"" originated as a pejorative description. Giorgio Vasari used the term ""barbarous German style"" in his Lives of the Artists to describe what is now considered the Gothic style, and in the introduction to the Lives he attributes various architectural features to ""the Goths"" whom he holds responsible for destroying the ancient buildings after they conquered Rome, and erecting new ones in this style. At the time in which Vasari was writing, Italy had experienced a century of building in the Classical architectural vocabulary revived in the Renaissance and seen as evidence of a new Golden Age of learning and refinement.",How long had Italy stopped building in a Classical architecture style at the time of Vasari?,"{'answer_start': [], 'text': []}"
4,Russian_Soviet_Federative_Socialist_Republic,"The international borders of the RSFSR touched Poland on the west; Norway and Finland on the northwest; and to its southeast were the Democratic People's Republic of Korea, Mongolian People's Republic, and the People's Republic of China. Within the Soviet Union, the RSFSR bordered the Ukrainian, Belarusian, Estonian, Latvian and Lithuanian SSRs to its west and Azerbaijan, Georgian and Kazakh SSRs to the south.",Which countries did the RSFSR conquer on the northwest?,"{'answer_start': [], 'text': []}"
5,Windows_

In [ ]:
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

In [ ]:
tokenizer("What is your name?", "I dont have a name")

{'input_ids': [101, 2054, 2003, 2115, 2171, 1029, 102, 1045, 2123, 2102, 2031, 1037, 2171, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
max_length = 512 # The maximum length of a feature (question and context)
doc_stride = 128 # The authorized overlap between two part of the context when splitting it is needed.

In [ ]:
for i, example in enumerate(datasets["train"]):
    if len(tokenizer(example["question"], example["context"])["input_ids"]) > max_length:
        break
example = datasets["train"][i]

In [ ]:
len(tokenizer(example["question"], example["context"])["input_ids"])

163

In [ ]:
pad_on_right = tokenizer.padding_side == "right"

In [ ]:
def prepare_train_features(examples):
    # Some of the questions have lots of whitespace on the left, which is not useful and will make the
    # truncation of the context fail (the tokenized question will take a lots of space). So we remove that
    # left whitespace
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # Tokenize our examples with truncation and padding, but keep the overflows using a stride. This results
    # in one example possible giving several features when a context is long, each of those features having a
    # context that overlaps a bit the context of the previous feature.
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Since one example might give us several features if it has a long context, we need a map from a feature to
    # its corresponding example. This key gives us just that.
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    # The offset mappings will give us a map from token to character position in the original context. This will
    # help us compute the start_positions and end_positions.
    offset_mapping = tokenized_examples.pop("offset_mapping")

    # Let's label those examples!
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        # We will label impossible answers with the index of the CLS token.
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Grab the sequence corresponding to that example (to know what is the context and what is the question).
        sequence_ids = tokenized_examples.sequence_ids(i)

        # One example can give several spans, this is the index of the example containing this span of text.
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        # If no answers are given, set the cls_index as answer.
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Start/end character index of the answer in the text.
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Start token index of the current span in the text.
            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1

            # End token index of the current span in the text.
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1

            # Detect if the answer is out of the span (in which case this feature is labeled with the CLS index).
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # Otherwise move the token_start_index and token_end_index to the two ends of the answer.
                # Note: we could go after the last offset if the answer is the last word (edge case).
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

In [ ]:
features = prepare_train_features(datasets['train'][:5])

In [ ]:
tokenized_datasets = datasets.map(prepare_train_features, batched=True, remove_columns=datasets["train"].column_names)

  0%|          | 0/62 [00:00<?, ?ba/s]

  0%|          | 0/14 [00:00<?, ?ba/s]

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
#wapas bulaunga ;-}

Downloading:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [ ]:
model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    "Electra-qa",
    evaluation_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    push_to_hub=True,
)

In [ ]:
from transformers import default_data_collator

data_collator = default_data_collator

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

Cloning https://huggingface.co/Sartc/Electra-qa into local empty directory.


In [ ]:
import gc
import torch
gc.collect()

torch.cuda.empty_cache()

In [ ]:
trainer.train()

/usr/local/lib/python3.8/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 61777
  Num Epochs = 2
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 7724
  Number of trainable parameters = 108893186


Epoch,Training Loss,Validation Loss


Saving model checkpoint to Electra-qa/checkpoint-500
Configuration saved in Electra-qa/checkpoint-500/config.json
Model weights saved in Electra-qa/checkpoint-500/pytorch_model.bin
tokenizer config file saved in Electra-qa/checkpoint-500/tokenizer_config.json
Special tokens file saved in Electra-qa/checkpoint-500/special_tokens_map.json
tokenizer config file saved in Electra-qa/tokenizer_config.json
Special tokens file saved in Electra-qa/special_tokens_map.json
Saving model checkpoint to Electra-qa/checkpoint-1000
Configuration saved in Electra-qa/checkpoint-1000/config.json
Model weights saved in Electra-qa/checkpoint-1000/pytorch_model.bin
tokenizer config file saved in Electra-qa/checkpoint-1000/tokenizer_config.json
Special tokens file saved in Electra-qa/checkpoint-1000/special_tokens_map.json
Saving model checkpoint to Electra-qa/checkpoint-1500
Configuration saved in Electra-qa/checkpoint-1500/config.json
Model weights saved in Electra-qa/checkpoint-1500/pytorch_model.bin
token

In [ ]:
trainer.push_to_hub()

In [ ]:
trainer.save_model("electro-interiit-trained")

In [ ]:
from transformers import Trainer

In [ ]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained("Sartc/electra-model-interiit-finetune-squad")

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
import torch

for batch in trainer.get_eval_dataloader():
    break
batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
with torch.no_grad():
    output = trainer.model(**batch)
output.keys()

In [ ]:
output.start_logits.shape, output.end_logits.shape

In [ ]:
output.start_logits.argmax(dim=-1), output.end_logits.argmax(dim=-1)

In [ ]:
n_best_size = 20

In [ ]:
import numpy as np

start_logits = output.start_logits[0].cpu().numpy()
end_logits = output.end_logits[0].cpu().numpy()
# Gather the indices the best start/end logits:
start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
valid_answers = []
for start_index in start_indexes:
    for end_index in end_indexes:
        if start_index <= end_index: # We need to refine that test to check the answer is inside the context
            valid_answers.append(
                {
                    "score": start_logits[start_index] + end_logits[end_index],
                    "text": "" # We need to find a way to get back the original substring corresponding to the answer in the context
                }
            )

In [ ]:
def prepare_validation_features(examples):
    # Some of the questions have lots of whitespace on the left, which is not useful and will make the
    # truncation of the context fail (the tokenized question will take a lots of space). So we remove that
    # left whitespace
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # Tokenize our examples with truncation and maybe padding, but keep the overflows using a stride. This results
    # in one example possible giving several features when a context is long, each of those features having a
    # context that overlaps a bit the context of the previous feature.
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Since one example might give us several features if it has a long context, we need a map from a feature to
    # its corresponding example. This key gives us just that.
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    # We keep the example_id that gave us this feature and we will store the offset mappings.
    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        # Grab the sequence corresponding to that example (to know what is the context and what is the question).
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0

        # One example can give several spans, this is the index of the example containing this span of text.
        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        # Set to None the offset_mapping that are not part of the context so it's easy to determine if a token
        # position is part of the context or not.
        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples["offset_mapping"][i])
        ]

    return tokenized_examples

In [ ]:
validation_features = datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=datasets["validation"].column_names
)

In [ ]:
raw_predictions = trainer.predict(validation_features)

In [ ]:
validation_features.set_format(type=validation_features.format["type"], columns=list(validation_features.features.keys()))

In [ ]:
max_answer_length = 30

In [ ]:
start_logits = output.start_logits[0].cpu().numpy()
end_logits = output.end_logits[0].cpu().numpy()
offset_mapping = validation_features[0]["offset_mapping"]
# The first feature comes from the first example. For the more general case, we will need to be match the example_id to
# an example index
context = datasets["validation"][0]["context"]

# Gather the indices the best start/end logits:
start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
valid_answers = []
for start_index in start_indexes:
    for end_index in end_indexes:
        # Don't consider out-of-scope answers, either because the indices are out of bounds or correspond
        # to part of the input_ids that are not in the context.
        if (
            start_index >= len(offset_mapping)
            or end_index >= len(offset_mapping)
            or offset_mapping[start_index] is None
            or offset_mapping[end_index] is None
        ):
            continue
        # Don't consider answers with a length that is either < 0 or > max_answer_length.
        if end_index < start_index or end_index - start_index + 1 > max_answer_length:
            continue
        if start_index <= end_index: # We need to refine that test to check the answer is inside the context
            start_char = offset_mapping[start_index][0]
            end_char = offset_mapping[end_index][1]
            valid_answers.append(
                {
                    "score": start_logits[start_index] + end_logits[end_index],
                    "text": context[start_char: end_char]
                }
            )

valid_answers = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[:n_best_size]
valid_answers

In [ ]:
datasets["validation"][0]["answers"]

In [ ]:
import collections

examples = datasets["validation"]
features = validation_features

example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
features_per_example = collections.defaultdict(list)
for i, feature in enumerate(features):
    features_per_example[example_id_to_index[feature["example_id"]]].append(i)

In [ ]:
from tqdm.auto import tqdm

def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size = 20, max_answer_length = 30):
    all_start_logits, all_end_logits = raw_predictions
    # Build a map example to its corresponding features.
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    # The dictionaries we have to fill.
    predictions = collections.OrderedDict()

    # Logging.
    print(f"Post-processing {len(examples)} example predictions split into {len(features)} features.")

    # Let's loop over all the examples!
    for example_index, example in enumerate(tqdm(examples)):
        # Those are the indices of the features associated to the current example.
        feature_indices = features_per_example[example_index]

        min_null_score = None # Only used if squad_v2 is True.
        valid_answers = []
        
        context = example["context"]
        # Looping through all the features associated to the current example.
        for feature_index in feature_indices:
            # We grab the predictions of the model for this feature.
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            # This is what will allow us to map some the positions in our logits to span of texts in the original
            # context.
            offset_mapping = features[feature_index]["offset_mapping"]

            # Update minimum null prediction.
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or min_null_score < feature_null_score:
                min_null_score = feature_null_score

            # Go through all possibilities for the `n_best_size` greater start and end logits.
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Don't consider out-of-scope answers, either because the indices are out of bounds or correspond
                    # to part of the input_ids that are not in the context.
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    # Don't consider answers with a length that is either < 0 or > max_answer_length.
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char: end_char]
                        }
                    )
        
        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            # In the very rare edge case we have not a single non-null prediction, we create a fake prediction to avoid
            # failure.
            best_answer = {"text": "", "score": 0.0}
        
        # Let's pick our final answer: the best one or the null answer (only for squad_v2)
        if not squad_v2:
            predictions[example["id"]] = best_answer["text"]
        else:
            answer = best_answer["text"] if best_answer["score"] > min_null_score else ""
            predictions[example["id"]] = answer

    return predictions

In [ ]:
start_indexes

In [ ]:
final_predictions = postprocess_qa_predictions(datasets["validation"], validation_features, raw_predictions.predictions)

In [ ]:
metric = load_metric("squad_v2")

In [ ]:
metric

In [ ]:
#if squad_v2:
formatted_predictions = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()]
#else:
#    formatted_predictions = [{"id": k, "prediction_text": v} for k, v in final_predictions.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in datasets["validation"]]
metric.compute(predictions=formatted_predictions, references=references)

In [ ]:
num = 20
for i in range(min(num, len(references))):
  id = references[i]["id"]
  pred_text = formatted_predictions[i]["prediction_text"]
  real_text = references[i]["answers"]["text"]
  print("id: " + str(id))
  print("ground truth: " + str(real_text))
  print("predicted_text: " + str(pred_text))
  print("\n")